# General setup

In [1]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [2]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [3]:
# Setup
model_path = './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/' # './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/'  './models/Interpolation_v2_Unitcell_latentLog_2d_latentMSE_biggerDecoder/'
model_name = 'best_model_annealed.pth' # 'best_model_annealed.pth'    'best_model.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# latent_space_version = '_prior' # '' or '_prior'

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [5]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [6]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
    df_rec[['ls1_prior', 'ls2_prior']] = df_rec['latent_space_mean_prior'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_rec['latent_space_mean_prior'].values.tolist()))

In [7]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-7.497217,0.000447,0.000201,0.000243,2.899288e-06,0.000104,RheniumTrioxide,11.697360
1,-4.972597,0.004762,0.004450,0.000312,4.065881e-07,0.002105,Wurtzite,22.798485
2,-5.280570,0.004465,0.002384,0.002081,2.128737e-08,0.000585,CaesiumChloride,14.635876
3,-7.253907,0.000654,0.000364,0.000288,1.977553e-06,0.000052,RheniumTrioxide,53.601151
4,-7.428283,0.000542,0.000243,0.000298,8.919368e-07,0.000051,RheniumTrioxide,11.694450


In [8]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,RheniumTrioxide,56,36,20,"[3.9052395820617676, 3.917825937271118, 3.8016...","[[0.006060000043362379, 0.007969999685883522, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[-0.3165692389011383, 0.9069648385047913]","[0.05926262214779854, 0.08797002583742142]","[-0.31705886125564575, 0.9077787399291992]","[0.06034988537430763, 0.08983772993087769]","[-0.3182690441608429, -0.3182690441608429, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[73, 8, 8, 8, 73, 73, 73, 8, 8, 8, 8, 8, 8, 73...",-0.316569,0.906965,-0.317059,0.907779
1,Wurtzite,56,25,31,"[3.218722343444824, 3.1993985176086426, 5.0874...","[[0.338919997215271, 0.6596699953079224, 0.011...","[2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, ...","[0.20500552654266357, 0.7467686533927917]","[0.05261911079287529, 0.0746200755238533]","[0.21350525319576263, 0.7401646375656128]","[0.052947383373975754, 0.0746118351817131]","[-0.652172863483429, -0.652172863483429, -0.09...","[[0.3333300054073334, 0.666670024394989, 0.0],...","[52, 8, 52, 8, 52, 52, 8, 8, 52, 52, 52, 8, 8,...",0.205006,0.746769,0.213505,0.740165
2,CaesiumChloride,56,20,36,"[3.235556125640869, 3.224201202392578, 3.46702...","[[0.02751000039279461, 0.013829999603331089, -...","[2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, ...","[1.139326572418213, -0.1244354248046875]","[0.08449804782867432, 0.1010625958442688]","[1.1338622570037842, -0.11764539778232574]","[0.0861816331744194, 0.10187134891748428]","[-0.6470714807510376, -0.6470714807510376, -0....","[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.0, 0.0, ...","[56, 8, 56, 56, 56, 56, 56, 56, 8, 8, 8, 56, 5...",1.139327,-0.124435,1.133862,-0.117645
3,RheniumTrioxide,56,36,20,"[3.8932340145111084, 3.9084558486938477, 3.761...","[[0.004279999993741512, 0.008089999668300152, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[-0.3201090097427368, 0.9113876223564148]","[0.05960838124155998, 0.08833424746990204]","[-0.32012271881103516, 0.9104672074317932]","[0.06029162555932999, 0.08970079571008682]","[-0.31839659810066223, -0.31839659810066223, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[22, 8, 8, 8, 22, 22, 22, 8, 8, 8, 8, 8, 8, 22...",-0.320109,0.911388,-0.320123,0.910467
4,RheniumTrioxide,56,36,20,"[3.8739497661590576, 3.876438856124878, 3.7988...","[[0.0006900000153109431, 0.005119999870657921,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[-0.31219246983528137, 0.908652126789093]","[0.05706716328859329, 0.08509664982557297]","[-0.3124615550041199, 0.9096876382827759]","[0.05773293972015381, 0.08627597987651825]","[-0.31885820627212524, -0.31885820627212524, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[31, 8, 8, 8, 31, 31, 31, 8, 8, 8, 8, 8, 8, 31...",-0.312192,0.908652,-0.312462,0.909688


In [9]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-7.497217,0.000447,0.000201,0.000243,2.899288e-06,0.000104,11.697360,RheniumTrioxide,56,36,...,"[0.05926262214779854, 0.08797002583742142]","[-0.31705886125564575, 0.9077787399291992]","[0.06034988537430763, 0.08983772993087769]","[-0.3182690441608429, -0.3182690441608429, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[73, 8, 8, 8, 73, 73, 73, 8, 8, 8, 8, 8, 8, 73...",-0.316569,0.906965,-0.317059,0.907779
1,-4.972597,0.004762,0.004450,0.000312,4.065881e-07,0.002105,22.798485,Wurtzite,56,25,...,"[0.05261911079287529, 0.0746200755238533]","[0.21350525319576263, 0.7401646375656128]","[0.052947383373975754, 0.0746118351817131]","[-0.652172863483429, -0.652172863483429, -0.09...","[[0.3333300054073334, 0.666670024394989, 0.0],...","[52, 8, 52, 8, 52, 52, 8, 8, 52, 52, 52, 8, 8,...",0.205006,0.746769,0.213505,0.740165
2,-5.280570,0.004465,0.002384,0.002081,2.128737e-08,0.000585,14.635876,CaesiumChloride,56,20,...,"[0.08449804782867432, 0.1010625958442688]","[1.1338622570037842, -0.11764539778232574]","[0.0861816331744194, 0.10187134891748428]","[-0.6470714807510376, -0.6470714807510376, -0....","[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.0, 0.0, ...","[56, 8, 56, 56, 56, 56, 56, 56, 8, 8, 8, 56, 5...",1.139327,-0.124435,1.133862,-0.117645
3,-7.253907,0.000654,0.000364,0.000288,1.977553e-06,0.000052,53.601151,RheniumTrioxide,56,36,...,"[0.05960838124155998, 0.08833424746990204]","[-0.32012271881103516, 0.9104672074317932]","[0.06029162555932999, 0.08970079571008682]","[-0.31839659810066223, -0.31839659810066223, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[22, 8, 8, 8, 22, 22, 22, 8, 8, 8, 8, 8, 8, 22...",-0.320109,0.911388,-0.320123,0.910467
4,-7.428283,0.000542,0.000243,0.000298,8.919368e-07,0.000051,11.694450,RheniumTrioxide,56,36,...,"[0.05706716328859329, 0.08509664982557297]","[-0.3124615550041199, 0.9096876382827759]","[0.05773293972015381, 0.08627597987651825]","[-0.31885820627212524, -0.31885820627212524, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[31, 8, 8, 8, 31, 31, 31, 8, 8, 8, 8, 8, 8, 31...",-0.312192,0.908652,-0.312462,0.909688


In [10]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

Index(['epoch', 'posterior_mean', 'posterior_std', 'prior_mean', 'prior_std',
       'np_size', 'crystal_type', 'crystal_system', 'space_group', 'ls1',
       'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')


## Plot

In [11]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [12]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [13]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [14]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [15]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [16]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [17]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [18]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [19]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958046674728394, -1.4048123359680176]","[0.12149954587221146, 0.17361588776111603]","[0.16966724395751953, -1.524801254272461]","[4.609440803527832, 4.613247871398926, 2.88265...","[[0.01720000058412552, -0.01331000030040741, 0...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.349580,-1.404812
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034023851156235, 0.14894968271255493]","[-0.8966662287712097, 0.11751437187194824]","[8.642404556274414, 8.638813972473145, 8.59504...","[[0.5150099992752075, 0.5133799910545349, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.899346,0.109892
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640621185302734, -1.4373685121536255]","[0.11025752872228622, 0.1611802875995636]","[0.5369250774383545, -1.4348397254943848]","[4.49102783203125, 4.481672763824463, 2.867542...","[[0.005669999867677689, 0.006409999914467335, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.356406,-1.437369
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.4590970277786255, -0.8510302901268005]","[0.10611193627119064, 0.1737586259841919]","[-0.3213590085506439, -0.6494584083557129]","[5.30486536026001, 5.314239025115967, 5.279217...","[[-0.0047900001518428326, -0.00584000023081898...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.459097,-0.851030
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.46544915437698364, -0.853926956653595]","[0.10532078146934509, 0.17308756709098816]","[-0.1431400179862976, -1.0178743600845337]","[5.941473960876465, 5.934543132781982, 5.91340...","[[-0.028349999338388443, 0.037870001047849655,...","[2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, ...",-0.465449,-0.853927


In [20]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [21]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [22]:
# Plot the experimental data
# index_list = [0,1,2] # Rutile samples
index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

# Interpolate in latent space

## Code

In [23]:
# Create latent samples for interpolation between two points
n_latent_samples = 7

# Define two latent points from pca space
# latent_point_1 = (24, 1) # ex1 (9, -13)
# latent_point_2 = (24, -15) # ex1 (24,-9)

# Define n latent points from latent space
crystal_types = ['NickelArsenide', 'CadmiumIodide'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']  ['NickelArsenide', 'CadmiumIodide', 'CadmiumChloride']
indices = [0, 4] #[14,3] #[0, 4, 0]  [0, 4, 3]
point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [24]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [25]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp['Interpolation number'] = df_interp['name'].apply(lambda x: int(x.split('_')[1]))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2,Interpolation number
0,sample_0,"[0.8116130828857422, 0.5675083994865417]","[3.057211399078369, 3.043759822845459, 5.24590...","[[0.007379999849945307, -0.013700000010430813,...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.811613,0.567508,0
1,sample_1,"[0.7867078185081482, 0.7122237682342529]","[2.915699005126953, 2.900284767150879, 5.16323...","[[0.012409999966621399, -0.007790000177919865,...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.786708,0.712224,1
2,sample_2,"[0.7618025541305542, 0.8569391369819641]","[2.629938840866089, 2.6032485961914062, 5.0948...","[[-0.0048699998296797276, -0.02347999997437000...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.761803,0.856939,2
3,sample_3,"[0.7368972301483154, 1.0016545057296753]","[2.337993860244751, 2.326245069503784, 4.94977...","[[-0.00892999954521656, -0.03223000094294548, ...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.736897,1.001655,3
4,sample_4,"[0.7119919657707214, 1.1463698148727417]","[2.627181053161621, 2.6396384239196777, 4.4853...","[[-0.002950000111013651, -0.02377999946475029,...","[2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, ...",0.711992,1.146370,4


## Plot

In [26]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [27]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0, zorder=10, lw=2)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1, zorder=10, lw=2)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
line_1 = 3.5
line_2 = 4.5
line_3 = 8.5
line_4 = 10.5
ax0.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax0.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)
ax1.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [28]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'CadmiumIodide'
crystal_type_index = 0
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean_prior.iloc[dist_index]
dist_std = df_rec.latent_space_std_prior.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [29]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [30]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,Dist. mean,"[0.6599715948104858, 1.422789454460144]","[2.970757246017456, 2.9796652793884277, 4.2430...","[[0.0014100000262260437, -0.003120000008493662...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.659972,1.422789
1,sample_1,"[0.6116375923156738, 1.4281234741210938]","[2.968167304992676, 2.978377342224121, 4.21416...","[[0.0006000000284984708, -0.003930000122636557...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.611638,1.428123
2,sample_2,"[0.6580829620361328, 1.3919413089752197]","[2.9578654766082764, 2.9683995246887207, 4.219...","[[0.0024999999441206455, -0.003379999892786145...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.658083,1.391941
3,sample_3,"[0.5417986512184143, 1.3211859464645386]","[2.934375047683716, 2.945712089538574, 4.12595...","[[0.00394000019878149, -0.003289999905973673, ...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.541799,1.321186
4,sample_4,"[0.6066400408744812, 1.446785569190979]","[2.9776711463928223, 2.9868645668029785, 4.224...","[[0.0001500000071246177, -0.003650000086054206...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.606640,1.446786


## Plot

In [31]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sampling_samples.pdf', dpi=300)

# Sample every latent distrubution and calculate loss

In [32]:
# Number of samples per point
n_samples = 100 #1000
test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    cell_positions_padded = np.zeros((setup_json['model']['out_dim'], 3))
    cell_positions_padded[:len(df_rec['cell_positions'].iloc[data_index]), :] = np.array(df_rec['cell_positions'].iloc[data_index])
    # if setup_json['training']['simplified_atom_identities']:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 3))
    # else:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 119))
    cell_atoms_padded = np.zeros((setup_json['model']['out_dim'],))
    cell_atoms_padded[:len(df_rec['cell_atoms'].iloc[data_index])] = np.array(df_rec['cell_atoms'].iloc[data_index])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([cell_positions_padded]*(n_samples+1))
    ground_truth_cell_atoms.extend([cell_atoms_padded]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters, dtype=torch.float32)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions, dtype=torch.float32)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms, dtype=torch.long)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [33]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [34]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for batch_index, sample_index in enumerate(sample_indeces):
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        if data_crystal_types[sample_index] == 'Interpolated':
            sample_results['crystalType'].append(data_crystal_types[sample_index])#+ ' ' + str(sum(cell_atoms_true[batch_index] > 0).item()))
        else:
            sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [35]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,RheniumTrioxide,"[-0.3165692389011383, 0.9069648385047913]","[3.8767147064208984, 3.8805129528045654, 3.781...","[[0.001069999998435378, 0.004350000061094761, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.011061,-0.316569,0.906965
1,0_0,sample,RheniumTrioxide,"[-0.20237518846988678, 1.0378012657165527]","[3.91549015045166, 3.9296815395355225, 4.07543...","[[0.0173799991607666, 0.02151999995112419, 0.0...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.337088,-0.202375,1.037801
2,0_1,sample,RheniumTrioxide,"[-0.2631903886795044, 0.7217420935630798]","[3.9053311347961426, 3.8761708736419678, 3.867...","[[-0.0063599999994039536, -0.00294000003486871...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.123595,-0.263190,0.721742
3,0_2,sample,RheniumTrioxide,"[-0.27636438608169556, 0.7983618974685669]","[3.882750988006592, 3.8746180534362793, 3.8279...","[[0.0010300000431016088, 0.0032999999821186066...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.092301,-0.276364,0.798362
4,0_3,sample,RheniumTrioxide,"[-0.3191215395927429, 0.765802264213562]","[3.885605812072754, 3.885427474975586, 3.88311...","[[0.001120000029914081, 0.004650000017136335, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.112601,-0.319122,0.765802


In [36]:
# Plot sampling results in latent space
if 'Interpolation' in model_path:
    custom_palette = ['k', sns.color_palette('tab20')[4], sns.color_palette('tab20')[7]]
    custom_markers = ['o', (4,0,0), (4,1,45)]
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & (df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette[1:], markers=custom_markers[1:])
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & ~(df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', marker='o', s=50, palette='viridis')
        
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
else:
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.svg', dpi=300, transparent=True)

In [37]:
# Plot interpolation results over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.svg', dpi=300, transparent=True)

In [38]:
df_exp_results_2 = df_exp_results.copy()
df_exp_results_2['Label'] = ''
for i in range(len(df_exp_results_2)):
    if df_exp_results_2['composition'].iloc[i] == 'IrO2':
        df_exp_results_2['Label'].iloc[i] = 'IrO2 (Rutile)'
    elif df_exp_results_2['composition'].iloc[i] == 'CeO2':
        df_exp_results_2['Label'].iloc[i] = 'CeO2 (Fluorite)'
    else:
        df_exp_results_2['Label'].iloc[i] = 'HEA (Spinel)'
        
# # Remove index 1
# df_exp_results_2.drop(index=1, inplace=True)

In [39]:
# Plot experimental data over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='ls1', y='ls2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='pc1', y='pc2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_sample_density.pdf', dpi=300)

In [40]:
df_exp_results

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2,ground_truth_crystal_type
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958046674728394, -1.4048123359680176]","[0.12149954587221146, 0.17361588776111603]","[0.16966724395751953, -1.524801254272461]","[4.609440803527832, 4.613247871398926, 2.88265...","[[0.01720000058412552, -0.01331000030040741, 0...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.349580,-1.404812,Rutile
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034023851156235, 0.14894968271255493]","[-0.8966662287712097, 0.11751437187194824]","[8.642404556274414, 8.638813972473145, 8.59504...","[[0.5150099992752075, 0.5133799910545349, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.899346,0.109892,Rutile
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640621185302734, -1.4373685121536255]","[0.11025752872228622, 0.1611802875995636]","[0.5369250774383545, -1.4348397254943848]","[4.49102783203125, 4.481672763824463, 2.867542...","[[0.005669999867677689, 0.006409999914467335, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.356406,-1.437369,Rutile
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.4590970277786255, -0.8510302901268005]","[0.10611193627119064, 0.1737586259841919]","[-0.3213590085506439, -0.6494584083557129]","[5.30486536026001, 5.314239025115967, 5.279217...","[[-0.0047900001518428326, -0.00584000023081898...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.459097,-0.851030,Fluorite
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.46544915437698364, -0.853926956653595]","[0.10532078146934509, 0.17308756709098816]","[-0.1431400179862976, -1.0178743600845337]","[5.941473960876465, 5.934543132781982, 5.91340...","[[-0.028349999338388443, 0.037870001047849655,...","[2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, ...",-0.465449,-0.853927,Fluorite
5,CeO2,"[[0.0], [0.0007327766506932676], [0.0014382996...","[-0.4588150978088379, -0.8527802228927612]","[0.10431147366762161, 0.1725536286830902]","[-0.4064563512802124, -0.9077193140983582]","[5.246377468109131, 5.269467353820801, 5.20481...","[[0.00215000007301569, 0.0017900000093504786, ...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.458815,-0.852780,Fluorite
6,Cu0.5Ni0.5Cr0.66Mn0.66Fe0.66O4,"[[0.0], [-1.6920250345719978e-05], [-3.3054449...","[-0.7703374028205872, 0.14266054332256317]","[0.056033581495285034, 0.1240103617310524]","[-0.7013264298439026, -0.12512458860874176]","[7.851334571838379, 7.837592124938965, 7.73774...","[[0.4024600088596344, 0.39546000957489014, 0.3...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.770337,0.142661,Spinel
7,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [0.001129354815930128], [0.00220868061...","[-0.7724447250366211, 0.15924935042858124]","[0.05540243908762932, 0.1225060224533081]","[-0.7790066599845886, 0.1722516119480133]","[8.33822250366211, 8.331743240356445, 8.300750...","[[0.5146999955177307, 0.51569002866745, 0.5024...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.772445,0.159249,Spinel
8,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [5.6757980928523466e-05], [0.000110909...","[-0.7670082449913025, 0.16185835003852844]","[0.05355964973568916, 0.11947672814130783]","[-0.7890729308128357, 0.0502982959151268]","[8.386552810668945, 8.382131576538086, 8.32871...","[[0.5119400024414062, 0.5091099739074707, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.767008,0.161858,Spinel
9,Co0.33Cu0.33Ni0.33CrMnO4,"[[0.0], [-1.0808051229105331e-05], [-2.1100040...","[-0.7618623375892639, 0.14131325483322144]","[0.05590708181262016, 0.12357873469591141]","[-0.7103788256645203, -0.021162331104278564]","[8.112070083618164, 8.09263801574707, 8.005455...","[[0.481440007686615, 0.4765399992465973, 0.477...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.761862,0.141313,Spinel


In [41]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,RheniumTrioxide,"[-0.3165692389011383, 0.9069648385047913]","[3.8767147064208984, 3.8805129528045654, 3.781...","[[0.001069999998435378, 0.004350000061094761, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.011061,-0.316569,0.906965
1,0_0,sample,RheniumTrioxide,"[-0.20237518846988678, 1.0378012657165527]","[3.91549015045166, 3.9296815395355225, 4.07543...","[[0.0173799991607666, 0.02151999995112419, 0.0...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.337088,-0.202375,1.037801
2,0_1,sample,RheniumTrioxide,"[-0.2631903886795044, 0.7217420935630798]","[3.9053311347961426, 3.8761708736419678, 3.867...","[[-0.0063599999994039536, -0.00294000003486871...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.123595,-0.263190,0.721742
3,0_2,sample,RheniumTrioxide,"[-0.27636438608169556, 0.7983618974685669]","[3.882750988006592, 3.8746180534362793, 3.8279...","[[0.0010300000431016088, 0.0032999999821186066...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.092301,-0.276364,0.798362
4,0_3,sample,RheniumTrioxide,"[-0.3191215395927429, 0.765802264213562]","[3.885605812072754, 3.885427474975586, 3.88311...","[[0.001120000029914081, 0.004650000017136335, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.112601,-0.319122,0.765802


In [42]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [43]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,RheniumTrioxide,"[-0.3165692389011383, 0.9069648385047913]","[3.8767147064208984, 3.8805129528045654, 3.781...","[[0.001069999998435378, 0.004350000061094761, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.011061,-0.316569,0.906965
1,0_0,sample,RheniumTrioxide,"[-0.20237518846988678, 1.0378012657165527]","[3.91549015045166, 3.9296815395355225, 4.07543...","[[0.0173799991607666, 0.02151999995112419, 0.0...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.337088,-0.202375,1.037801
2,0_1,sample,RheniumTrioxide,"[-0.2631903886795044, 0.7217420935630798]","[3.9053311347961426, 3.8761708736419678, 3.867...","[[-0.0063599999994039536, -0.00294000003486871...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.123595,-0.263190,0.721742
3,0_2,sample,RheniumTrioxide,"[-0.27636438608169556, 0.7983618974685669]","[3.882750988006592, 3.8746180534362793, 3.8279...","[[0.0010300000431016088, 0.0032999999821186066...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.092301,-0.276364,0.798362
4,0_3,sample,RheniumTrioxide,"[-0.3191215395927429, 0.765802264213562]","[3.885605812072754, 3.885427474975586, 3.88311...","[[0.001120000029914081, 0.004650000017136335, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.112601,-0.319122,0.765802


In [44]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()

# Interpolation dataset input to model

In [45]:
# Load CHILI dataset
dataset_interp = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation_v2',
    graph_type=setup_json['data']['graph_type'],
)

# Load/create data splits
try:
    # Load existing data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
except FileNotFoundError:
    # Create new data split
    dataset_interp.create_data_split(
        test_size=setup_json['data']['split']['test'],
        validation_size=setup_json['data']['split']['validation'],
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
    # Load data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
# Dataloader
data_loader_interp = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [ ]:
# Load CHILI dataset
dataset_interp_uc = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation_v2',
    graph_type='unit_cell',
)

# Load existing data split
dataset_interp_uc.load_data_split(
    split_strategy=setup_json['data']['split_strategy'],
    stratify_on=setup_json['data']['stratify_column'],
)
    
# Dataloader
dataset_interp_uc = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [46]:
interp_data_results = {
    'name': [],
    'crystal_type': [],
    'pdf': [],
    'n_atoms': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_atoms': [],
    'cell_positions': [],
}
    

In [47]:
# Inference
model.eval()
for batch in tqdm(data_loader_interp, desc='Inference', disable=setup_json['disable_tqdm']):
    batch = batch.to(device)
    this_batch_size = batch.batch.amax().item() + 1
    
    # n_atoms = batch.y['unit_cell_n_atoms']
    n_atoms = batch.y['n_atoms']
    
    # Normalize scattering
    pdf = batch.y['xPDF'][:,1,:].unsqueeze(-1)
    if setup_json['data']['normalize_scattering']:
        # Normalize so highest peak in each sample is 1
        pdf /= torch.amax(pdf, dim=1, keepdim=True)[0]
    
    # Composition conditioning
    composition = torch.zeros(this_batch_size, 119).to(device)
    elements_in_batch = batch.y['atomic_species'].long()
    index_counter = 0
    for i in range(this_batch_size):
        n_elements = batch.y['n_atomic_species'][i]
        composition[i, elements_in_batch[index_counter:index_counter + n_elements]] = 1
        index_counter += n_elements
    composition[:, 0] = 1 
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
        
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store names
    interp_data_results['name'].extend(batch.data_id)
    interp_data_results['crystal_type'].extend(batch.y['crystal_type'])
    
    # Store PDF
    interp_data_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store number of atoms
    interp_data_results['n_atoms'].extend(n_atoms.cpu().tolist())
    
    # Store latent representation
    interp_data_results['prior_mean'].extend(prior_mean.cpu().tolist())
    interp_data_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    interp_data_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    interp_data_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_data_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_data_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # # Create CIF files
    # for batch_index in range(this_batch_size):
    #     # Prediction
    #     try:
    #         create_cif(
    #             cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
    #             cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
    #             cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
    #             filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
    #             prediction=True,
    #             composition=data_composition_string[composition_string_index[batch_index]],
    #             simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
    #         )
    #     except:
    #         print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_interp_data_results = pd.DataFrame(interp_data_results)
if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    df_interp_data_results[['ls1', 'ls2']] = df_interp_data_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_interp_data_results[['pc1', 'pc2']] = pca.transform(np.array(df_interp_data_results['prior_mean'].values.tolist()))
    
df_interp_data_results['Step'] = 0
df_interp_data_results['Step'][df_interp_data_results['crystal_type'] == 'interpolated'] = df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated']['name'].str.split('_').str[-3].str[-1].astype(int)

df_interp_data_results.head()

,name,crystal_type,pdf,n_atoms,prior_mean,prior_log_std,z_sample,cell_parameters,cell_atoms,cell_positions,ls1,ls2,Step
0,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.001080057700164616], [-0.002175908...",32,"[0.8141693472862244, 0.5546340346336365]","[0.07083693146705627, 0.08041650056838989]","[0.8279130458831787, 0.7284441590309143]","[2.912306308746338, 2.898115634918213, 5.17715...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.01018999982625246, -0.004129999782890081, ...",0.814169,0.554634,1
1,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0011973911896348], [-0.00233203941...",32,"[0.13067850470542908, 0.806233286857605]","[0.06508377939462662, 0.0903969332575798]","[0.11948082596063614, 0.8829857110977173]","[3.1979737281799316, 3.22377872467041, 5.13662...","[2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, ...","[[0.31779998540878296, 0.6359999775886536, 0.0...",0.130679,0.806233,4
2,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0011937911622226238], [-0.00241317...",32,"[0.8092480897903442, 0.6349810361862183]","[0.0702178031206131, 0.07984131574630737]","[0.6741210222244263, 0.687116265296936]","[2.925731658935547, 2.9048614501953125, 5.1149...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.0206300001591444, 0.008100000210106373, 0....",0.809248,0.634981,1
3,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0024064567405730486], [-0.00472889...",32,"[0.6902510523796082, 1.4623944759368896]","[0.07393131405115128, 0.07757934927940369]","[0.6422370672225952, 1.398971438407898]","[2.958134412765503, 2.969363212585449, 4.21487...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...","[[0.0021699999924749136, -0.00394000019878149,...",0.690251,1.462394,3
4,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0005925110308453441], [-0.00126495...",32,"[0.4385504126548767, -1.4993935823440552]","[0.0816754400730133, 0.14537791907787323]","[0.48167213797569275, -1.6848013401031494]","[4.5329389572143555, 4.528695583343506, 2.8801...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[[0.01940999925136566, 7.999999797903001e-05, ...",0.438550,-1.499394,3


## Plot

In [49]:
# Plot interpolation points in latent space

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='ls1', y='ls2', color='black', marker='o', s=50)
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset.pdf', dpi=300)


In [50]:
# Plot interpolation points in latent space (# of atoms)

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)

    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='n_atoms', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='# of atoms')
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)

    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset_nAtoms.pdf', dpi=300)